<a href="https://colab.research.google.com/github/ahtisham151/FYP-20-FL-24-Smart-Health-Monitoring/blob/main/LogReg_Hypertension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# https://www.figma.com/design/HkkxMcLwOBTkgdpJl7Bdnl/Vital-HealthCare-App?node-id=0-2&p=f

In [ ]:
# https://www.kaggle.com/datasets/prosperchuks/health-dataset/code

In [ ]:
# 📌 STEP 1: Imports and Setup
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import pickle

# 📌 STEP 2: Load Your Dataset (Upload in Colab)
from google.colab import files
uploaded = files.upload()

# Replace with your actual filename
df = pd.read_csv(next(iter(uploaded)))
df.head()

# 📌 STEP 3: Preprocess the Data
# Map categorical values if needed
df['sex'] = df['sex'].map({"male": 1, "female": 0}) if df['sex'].dtype == object else df['sex']

# Drop unnecessary columns
columns_to_drop = ['thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
df = df.drop(columns=columns_to_drop, errors='ignore')

# Drop rows with missing values for simplicity (can use imputation instead)
df = df.dropna()

# Separate features and target
X = df.drop(columns=['target'])
y = df['target']

# Standardize numeric features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 📌 STEP 4: Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 📌 STEP 5: Train Logistic Regression Model
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)

# 📌 STEP 6: Evaluate Model
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))
print("AUC-ROC:", roc_auc_score(y_test, y_proba))

# 📌 STEP 7: Save Model
# Save model in pickle format
with open("hypertension_logreg_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Download the model
files.download("hypertension_logreg_model.pkl")

# 📌 STEP 8: Make Prediction Function
def predict_patient(input_dict):
    input_df = pd.DataFrame([input_dict])
    input_scaled = scaler.transform(input_df)
    pred = model.predict(input_scaled)[0]
    prob = model.predict_proba(input_scaled)[0][1]
    return pred, prob

# Example
example_patient = {
    'age': 54,
    'sex': 1,
    'cp': 2,
    'trestbps': 140,
    'chol': 239,
    'fbs': 1,
    'restecg': 0
}

pred, prob = predict_patient(example_patient)
print("\nPrediction:", "Hypertension" if pred == 1 else "No Hypertension")
print("Probability of Hypertension:", round(prob, 3))


Saving hypertension_data.csv to hypertension_data.csv

Classification Report:

              precision    recall  f1-score   support

           0       0.68      0.78      0.73      2319
           1       0.80      0.71      0.75      2893

    accuracy                           0.74      5212
   macro avg       0.74      0.75      0.74      5212
weighted avg       0.75      0.74      0.74      5212

AUC-ROC: 0.7873007469070411


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Prediction: Hypertension
Probability of Hypertension: 0.621


In [ ]:
# Example
example_patient = {
    'age': 35,         # Younger age lowers risk
    'sex': 0,          # Female (statistically slightly lower risk)
    'cp': 0,           # No chest pain
    'trestbps': 115,   # Normal resting BP
    'chol': 180,       # Healthy cholesterol
    'fbs': 0,          # Normal fasting blood sugar
    'restecg': 0       # Normal ECG
}

pred, prob = predict_patient(example_patient)
print("\nPrediction:", "Hypertension" if pred == 1 else "No Hypertension")
print("Probability of Hypertension:", round(prob, 3))


Prediction: No Hypertension
Probability of Hypertension: 0.36
